In [ ]:
from pathlib import Path

import polars as pl
import polars.selectors as cs

In [ ]:
CSV_PATH = Path("polars_compare.csv")

In [ ]:
def load_comparison_lf(file_path: Path | None = None) -> pl.LazyFrame:
    csvpath = file_path if file_path is not None else CSV_PATH
    id_schema = {"id": pl.String, "timestamp": pl.Datetime}

    if not csvpath.is_file():
        lf = pl.LazyFrame({"id": [], "timestamp": []}, id_schema)
    else:
        lf = pl.scan_csv(csvpath, schema_overrides=id_schema, null_values="")

    schema = lf.collect_schema()
    match_col_names = [
        col_name for col_name in schema if ".is_match" in col_name
    ]

    match_cast_exprs = [
        pl.col(col_name).cast(pl.Boolean) for col_name in match_col_names
    ]
    lf = lf.with_columns(match_cast_exprs)

    return lf

In [ ]:
lf = load_comparison_lf()

In [ ]:
lf.collect_schema()

In [ ]:
lf.collect()

In [ ]:
schema = lf.collect_schema()
struct_col_names = [
    col_name
    for col_name, dtype in schema.items()
    if isinstance(dtype, pl.Struct)
]
match_lf = (
    lf.with_columns(
        pl.all_horizontal(cs.ends_with(".is_match") & cs.boolean())
        .fill_null(True)
        .alias("all_match")
    )
    .with_columns(
        pl.col("timestamp")
        .dt.convert_time_zone("America/Vancouver")
        .dt.to_string("%Y-%m-%d %H:%M:%S.%3f")
        .alias("last_run")
    )
    .sort("timestamp")
)

In [ ]:
last_by_id_lf = (
    match_lf.group_by("id")
    .last()
    .sort("id")
    .select(["id", "last_run", "all_match"])
)
# print(test_lf.explain())
last_by_id_lf.collect()

In [ ]:
bad_results_lf = last_by_id_lf.filter(~pl.col("all_match"))
bad_results_lf.collect()

In [ ]:
last_n = 5
last_n_runs = match_lf.tail(last_n).select(["id", "last_run", "all_match"])
last_n_runs.collect()